In [5]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()

# 데이터 로드 및 변환
timesteps = 13
features = 1

# CSV 데이터 불러오기
punch_data = pd.read_csv("punch.csv", usecols=range(13))
no_punch_data = pd.read_csv("normal.csv", usecols=range(13))
pet_data = pd.read_csv("pet.csv", usecols=range(13))
pinch_data = pd.read_csv("pinch.csv", usecols=range(13))

# NumPy 변환 후 float32 형 변환
punch_data = np.array(punch_data).astype(np.float32)
no_punch_data = np.array(no_punch_data).astype(np.float32)
pet_data = np.array(pet_data).astype(np.float32)
pinch_data = np.array(pinch_data).astype(np.float32)

# 샘플 개수
samples_punch = len(punch_data)
samples_no_punch = len(no_punch_data)
samples_pet = len(pet_data)
samples_pinch = len(pinch_data)

# 🔹 데이터 합치기 (Pinch 추가)
X_data = np.concatenate([punch_data, no_punch_data, pet_data, pinch_data], axis=0)  # (총 샘플 개수, 13)
# X_data_scaled = scaler.fit_transform(X_data)
X_data_scaled = scaler.fit_transform(X_data)


X_data = np.expand_dims(X_data_scaled, axis=-1)  # (총 샘플 개수, 13, 1)
print(X_data)
# 🔹 레이블 생성 (0: 펀치, 1: 비펀치, 2: 애완동물, 3: 핀치)
y_data = np.concatenate([
    np.zeros((samples_punch, 1)),  # 0: 펀치
    np.ones((samples_no_punch, 1)),  # 1: 비펀치
    np.full((samples_pet, 1), 2),  # 2: 쓰다듬기
    np.full((samples_pinch, 1), 3)  # 3: 핀치
], axis=0)

# PyTorch Tensor 변환
X_data = torch.tensor(X_data, dtype=torch.float32)
y_data = torch.tensor(y_data, dtype=torch.long).squeeze()

# 데이터셋 나누기
X_train, X_test, y_train, y_test = train_test_split(X_data, y_data, test_size=0.2, random_state=42)

# DataLoader 사용
batch_size = 4
train_dataset = TensorDataset(X_train, y_train)
test_dataset = TensorDataset(X_test, y_test)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size)

#  LSTM 모델 정의 (🔹 num_classes=4 로 변경)
class PunchDetectionLSTM(nn.Module):
    def __init__(self, num_classes=4):
        super(PunchDetectionLSTM, self).__init__()
        self.lstm = nn.LSTM(input_size=1, hidden_size=64, num_layers=2, batch_first=True)
        self.fc1 = nn.Linear(64, 32)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(32, num_classes)  # 🔹 4개 클래스 (펀치, 비펀치, 애완동물, 핀치)
    
    def forward(self, x):
        lstm_out, _ = self.lstm(x)
        x = self.fc1(lstm_out[:, -1, :])  # 마지막 타임스텝 사용
        x = self.relu(x)
        x = self.fc2(x)  # 🔹 Softmax 사용하지 않음 (CrossEntropyLoss 내에서 포함됨)
        return x

#  모델 학습
model = PunchDetectionLSTM()
criterion = nn.CrossEntropyLoss()  # 🔹 다중 분류를 위한 CrossEntropyLoss
optimizer = optim.Adam(model.parameters(), lr=0.001)

epochs = 15
for epoch in range(epochs):
    model.train()
    train_loss = 0.0

    for batch_X, batch_y in train_loader:
        optimizer.zero_grad()
        outputs = model(batch_X)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()

    # 검증 데이터 평가
    model.eval()
    val_loss = 0.0
    correct, total = 0, 0
    with torch.no_grad():
        for batch_X, batch_y in test_loader:
            outputs = model(batch_X)
            loss = criterion(outputs, batch_y)
            val_loss += loss.item()

            # 🔹 정확도 계산
            _, predicted = torch.max(outputs, 1)
            correct += (predicted == batch_y).sum().item()
            total += batch_y.size(0)

    accuracy = 100 * correct / total
    print(f"Epoch [{epoch+1}/{epochs}], Train Loss: {train_loss / len(train_loader):.4f}, "
          f"Val Loss: {val_loss / len(test_loader):.4f}, Accuracy: {accuracy:.2f}%")

# 모델 저장
torch.save(model.state_dict(), "please.pth")
print(" 모델이 저장되었습니다!")


[[[-0.46301743]
  [-0.26683813]
  [-0.3401693 ]
  ...
  [-0.6826781 ]
  [-0.5861136 ]
  [-0.50662446]]

 [[-0.44532317]
  [-0.56823516]
  [-0.45415065]
  ...
  [-0.6751483 ]
  [-0.54990155]
  [-0.473822  ]]

 [[-0.4672022 ]
  [-0.56823516]
  [-0.51974756]
  ...
  [-0.65644836]
  [-0.55256164]
  [-0.49252564]]

 ...

 [[ 1.0483632 ]
  [-0.44867545]
  [ 0.6974541 ]
  ...
  [ 1.1482394 ]
  [ 1.9212022 ]
  [ 2.3835652 ]]

 [[ 0.36144856]
  [-0.34809875]
  [-0.5522749 ]
  ...
  [ 1.6549121 ]
  [ 3.037896  ]
  [ 0.176666  ]]

 [[ 0.12682904]
  [-0.3890621 ]
  [-0.09445208]
  ...
  [ 6.239548  ]
  [ 0.42258787]
  [-0.26782337]]]
Epoch [1/15], Train Loss: 0.9628, Val Loss: 0.7023, Accuracy: 66.42%
Epoch [2/15], Train Loss: 0.6087, Val Loss: 0.4114, Accuracy: 86.13%
Epoch [3/15], Train Loss: 0.3527, Val Loss: 0.1869, Accuracy: 99.27%
Epoch [4/15], Train Loss: 0.2714, Val Loss: 0.0958, Accuracy: 97.81%
Epoch [5/15], Train Loss: 0.1431, Val Loss: 0.0852, Accuracy: 97.81%
Epoch [6/15], Train Loss: